## Standardise

In [ ]:
import pandas as pd
# Load the dataset
df = pd.read_csv("Gov Index.csv") 
# Convert columns to numeric in case of strings or missing values
df["GovIndex"] = pd.to_numeric(df["GovIndex"], errors="coerce")
df["Error"] = pd.to_numeric(df["Error"], errors="coerce")
# ---- 1. Add 2.5 to GovIndex ----
df["GovIndex"] = (df["GovIndex"] + 2.5)*20
# ---- 2. Convert standard error into percent ----
df["ErrorPercent"] = (df["Error"]/2.5) * 100

# Optional: Drop original Error column if you only want percent
df = df.drop(columns=["Error"])

# Preview
print(df.head())
df.to_csv("Gov Index2.csv",index=False)

   ISO  Year  GovIndex  percentile rank  ErrorPercent
0  AFG  1996      24.2              4.3          13.6
1  ALB  1996      32.2             19.4          12.8
2  DZA  1996      38.6             33.3          10.4
3  ASM  1996       NaN              NaN           NaN
4  ADO  1996      76.4             87.1          19.2


## Merge

In [ ]:
import pandas as pd

# Load datasets
gov = pd.read_csv("Gov Index2.csv")
gdp = pd.read_csv("GDP growth (annual %).csv")
# Ensure correct datatypes
gov["ISO"] = gov["ISO"].astype(str)
gov["Year"] = gov["Year"].astype(int)
gdp["ISO"] = gdp["ISO"].astype(str)
gdp["Year"] = gdp["Year"].astype(int)
# Perform inner join on ISO + Year
merged = gov.merge(
    gdp,
    on=["ISO", "Year"],    # keys match exactly
    how="inner"
)
print(gov.shape)
print()
print(merged.shape)
print()
print(merged.head())
print()
print(merged.info())
merged.to_csv("RSS_v2.csv", index=False)

(32100, 5)

(29850, 6)

   ISO  Year  GovIndex  percentile rank  ErrorPercent  \
0  AFG  1996      24.2              4.3          13.6   
1  ALB  1996      32.2             19.4          12.8   
2  DZA  1996      38.6             33.3          10.4   
3  ASM  1996       NaN              NaN           NaN   
4  AGO  1996      26.6              9.7          10.4   

   GDP per capita growth (annual %)  
0                               NaN  
1                          8.005514  
2                          2.081850  
3                               NaN  
4                          9.768938  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29850 entries, 0 to 29849
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ISO                               29850 non-null  object 
 1   Year                              29850 non-null  int32  
 2   GovIndex                          29211

In [8]:
hdi = pd.read_csv("Human Development Index.csv")
rest = pd.read_csv("RSS_v2.csv")  
# -----------------------------
# Convert HDI value to percent
# -----------------------------
hdi["value"] = pd.to_numeric(hdi["value"], errors="coerce")
# Create HDI percent column
hdi["HDI_percent"] = hdi["value"] * 100
# Drop original column if you only want the percent
hdi = hdi.drop(columns=["value"])
# -----------------------------
# Prepare join keys
# -----------------------------
hdi["ISO"] = hdi["ISO"].astype(str)
hdi["Year"] = hdi["Year"].astype(int)

rest["ISO"] = rest["ISO"].astype(str)
rest["Year"] = rest["Year"].astype(int)
# -----------------------------
# Inner join on ISO + Year
# -----------------------------
merged_all = rest.merge(
    hdi[["ISO", "Year", "HDI_percent"]],
    on=["ISO", "Year"],
    how="inner"
)
# -----------------------------
# Preview results
# -----------------------------
print(rest.shape)
print()
print(merged_all.shape)
print()
print(merged_all.head())
print(merged_all.info())
merged_all.to_csv("RSS_v3.csv", index=False)

(29850, 6)

(26970, 7)

   ISO  Year  GovIndex(%)  percentile rank  ErrorPercent  \
0  AFG  1996         24.2              4.3          13.6   
1  ALB  1996         32.2             19.4          12.8   
2  DZA  1996         38.6             33.3          10.4   
3  ARG  1996         48.0             53.8           8.4   
4  ARM  1996         40.6             38.2          13.6   

   GDP per capita growth (annual %)  HDI_percent  
0                               NaN         33.4  
1                          8.005514         64.7  
2                          2.081850         61.5  
3                          4.208050         76.0  
4                          6.796795         63.5  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26970 entries, 0 to 26969
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ISO                               26970 non-null  object 
 1   Year   

## Recovery Column Engineering

In [14]:
import numpy as np
# ---------------------------------------------------
# Load datasets
# ---------------------------------------------------
rss = pd.read_csv("RSS_v3.csv")     # macro data (all years)
dii = pd.read_csv("DII_v6.csv")     # disaster data (only disaster years)
# Ensure types
rss["ISO"] = rss["ISO"].astype(str)
rss["Year"] = rss["Year"].astype(int)
dii["ISO"] = dii["ISO"].astype(str)
dii["Year"] = dii["Year"].astype(int)
# Extract GDP growth into lookup table
gdp_lookup = rss.pivot_table(
    index="Year",
    columns="ISO",
    values="GDP per capita growth (annual %)",
    aggfunc="mean"
)

# ---------------------------------------------------
# Helper function to compute recovery years
# ---------------------------------------------------
def compute_recovery(iso, year):
    # ---------------------
    # 1. Compute baseline
    # ---------------------
    years_before = [year - 1, year - 2, year - 3]

    # extract available GDP growth values
    baseline_vals = []
    for y in years_before:
        try:
            val = gdp_lookup.loc[y, iso]
            if pd.notna(val):
                baseline_vals.append(val)
        except KeyError:
            # year not in index
            continue

    if len(baseline_vals) == 0:
        return np.nan

    baseline = np.mean(baseline_vals)
    # ---------------------
    # 2. Find recovery year
    # ---------------------
    for t in range(year + 1, 2025):     # up to and including 2024
        try:
            val = gdp_lookup.loc[t, iso]
        except KeyError:
            continue  # missing entire year → skip

        if pd.isna(val):
            continue  # missing data → skip

        if val >= baseline:            # recovery condition met
            return t - year
    # ---------------------
    # 3. If never recovered
    # ---------------------
    return 2024 - year
# ---------------------------------------------------
# Apply to each disaster row
# ---------------------------------------------------
# Work only with unique (ISO, Year) pairs from DII
# ---------------------------------------------------
disaster_pairs = dii[["ISO", "Year"]].drop_duplicates().copy()

# Compute recovery_years once per (ISO, Year)
disaster_pairs["recovery_years"] = disaster_pairs.apply(
    lambda row: compute_recovery(row["ISO"], row["Year"]),
    axis=1
)

# ---------------------------------------------------
# Merge results back with RSS_v3
# ---------------------------------------------------
final = rss.merge(
    disaster_pairs,   # NOTE: not full dii, just unique pairs
    on=["ISO", "Year"],
    how="left"
)
print(final.shape  )
print()
print(final.head())
print(final[final["recovery_years"].notna()].head())
print(final.info())
final.to_csv("RSS_v4.csv", index=False)

(26970, 8)

   ISO  Year  GovIndex(%)  percentile rank  GovIndex_ErrorPercent  \
0  AFG  1996         24.2              4.3                   13.6   
1  ALB  1996         32.2             19.4                   12.8   
2  DZA  1996         38.6             33.3                   10.4   
3  ARG  1996         48.0             53.8                    8.4   
4  ARM  1996         40.6             38.2                   13.6   

   GDP per capita growth (annual %)  HDI_percent  recovery_years  
0                               NaN         33.4             NaN  
1                          8.005514         64.7             NaN  
2                          2.081850         61.5             NaN  
3                          4.208050         76.0             NaN  
4                          6.796795         63.5             NaN  
     ISO  Year  GovIndex(%)  percentile rank  GovIndex_ErrorPercent  \
883  ALB  1998         30.2             18.2                    9.2   
885  ARG  1998         46.0  

## Add GDP-Post & Pre

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("RSS_v5.csv")
# Assume df has columns: "checker", "values"

# Initialize output columns
df["GDP_GrowthPost"] = np.nan
df["GDP_GrowthPre"] = np.nan

for i in range(len(df)):
    # Check condition: only compute when checker is non-null
    if pd.notna(df.loc[i, "recovery_years"]):
        # PRE windows: rows i-1, i-2, i-3
        pre_idx = [i-1, i-2, i-3]
        pre_vals = df.loc[
            [idx for idx in pre_idx if 0 <= idx < len(df)],
            "GDP per capita growth (annual %)"
        ]
        df.loc[i, "GDP_GrowthPre"] = pre_vals.mean()

        # POST windows: rows i+1, i+2, i+3
        post_idx = [i+1, i+2, i+3]
        post_vals = df.loc[
            [idx for idx in post_idx if 0 <= idx < len(df)],
            "GDP per capita growth (annual %)"
        ]
        df.loc[i, "GDP_GrowthPost"] = post_vals.mean()
df.head()
df.to_csv("RSS_v6.csv", index=False)

In [11]:
df =pd.read_csv("RSS_v6.csv")
print(df.shape)
print()
df = df.dropna(subset=['GDP per capita growth (annual %)'])
df = df.dropna(subset=['GDP_GrowthPost'])
df = df.dropna(subset=['recovery_years'])
df = df.dropna(subset=['Year'])
df = df.dropna(subset=['ISO'])
df = df.dropna(subset=['GDP_GrowthPre'])
df = df.dropna(subset=['HDI'])
df = df.dropna(subset=['GovIndex_ErrorPercent'])
df = df.dropna(subset=['GovIndex(%)'])
print(df.shape)
print()
df['RSS'] = ((df["GDP_GrowthPost"] - df["GDP_GrowthPre"] )/df['recovery_years'])+((df['HDI']+df['GovIndex_ErrorPercent'])/2)
df.to_csv("RSS_v7.csv")
print(df.shape)
print()

(26970, 11)

(16061, 11)

(16061, 12)



## Add country Name column

In [15]:
import pycountry
import pandas as pd
df = pd.read_csv('RSS_v9.csv')
def add_country_names(df):
    def iso_to_name(iso):
        try:
            return pycountry.countries.get(alpha_3=iso).name
        except:
            return None

    df["Country_Name"] = df["ISO"].apply(iso_to_name)
    return df
df =add_country_names(df)
print(df.head())
df.to_csv('RSS_v9.csv', index=False)

   ISO  Year  percentile rank  GovIndex  GDP per capita growth (annual %)  \
0  ALB  1998             18.2     -0.99                          8.993807   
1  ARG  1998             50.3     -0.20                          2.636541   
2  ARM  1998             21.9     -0.94                          8.567105   
3  AUS  1998             92.5      1.80                          3.625182   
4  AUT  1998             93.0      1.80                          3.379381   

   recovery_years    HDI  GDP_GrowthPost  GDP_GrowthPre       RSS Country  \
0               2  0.659        4.881050       6.093330 -0.771640     ALB   
1               5  0.772        5.190556       6.216656  0.080780     ARG   
2               4  0.655        5.321817       5.023284 -0.067867     ARM   
3               2  0.891        5.191604       4.881050  1.500777     AUS   
4               2  0.870        4.753567       4.942943  1.240312     AUT   

  Country_Name  
0      Albania  
1    Argentina  
2      Armenia  
3    A

## Data Merging

In [12]:
import pandas as pd
def clean_and_merge(file1, file2, output_file):
    # Load both CSVs
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    print(f"Shape of df1-{file1} before cleaning:", df1.shape)
    print(f"Shape of df2-{file2} before cleaning:", df2.shape)
    # Drop any rows with missing values
    df1_clean = df1[df1['Year']>=1998]
    df2_clean = df2[df2['Year']>=1998]
    print(f"Shape of df1-{file1} after dropna:", df1_clean.shape)
    print(f"Shape of df2-{file2} after dropna:", df2_clean.shape)
    df1_duplicates = df1_clean[df1_clean.duplicated(subset=["Year", "Country"], keep=False)]
    df2_duplicates = df2_clean[df2_clean.duplicated(subset=["Year", "Country"], keep=False)]

    print("df1 duplicate keys:", df1_duplicates.shape)
    print("df2 duplicate keys:", df2_duplicates.shape)

    # Inner join on 'Year' and 'Country'
    merged = df1_clean.merge(
        df2_clean,
        on=["Year", "Country"],
        how="inner"
    )
    print(f"Shape of merged-{output_file} dataframe:", merged.shape)
    # Save to output CSV
    merged.to_csv(output_file, index=False)
    print(f"Merged file saved as: {output_file}")
clean_and_merge('DII_v5.csv', 'RSS_v9.csv', 'RSS_v10.csv')
df = pd.read_csv('RSS_v10.csv')
print(df.shape)
print()
df = df.dropna()
print(df.shape)

Shape of df1-DII_v5.csv before cleaning: (16099, 16)
Shape of df2-RSS_v9.csv before cleaning: (16061, 11)
Shape of df1-DII_v5.csv after dropna: (10931, 16)
Shape of df2-RSS_v9.csv after dropna: (16061, 11)
df1 duplicate keys: (9679, 16)
df2 duplicate keys: (16061, 11)
Shape of merged-RSS_v10.csv dataframe: (54492, 25)
Merged file saved as: RSS_v10.csv
(54492, 25)

(1188, 25)


In [15]:
import pandas as pd

def clean_and_merge1(file1, file2, output_file):

    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    print("df1:", df1.shape)
    print("df2:", df2.shape)

    # Assign row numbers inside each Country-Year group
    df1["ID"] = df1.groupby(["Year", "Country"]).cumcount()
    df2["ID"] = df2.groupby(["Year", "Country"]).cumcount()

    # Merge ONLY rows where ID exists in both
    merged = df1.merge(
        df2,
        on=["Year", "Country", "ID"],
        how="inner",
        suffixes=("_DII", "_RSS")
    )

    print("Merged shape:", merged.shape)

    merged.to_csv(output_file, index=False)
    print(f"Saved as {output_file}")
clean_and_merge1("DII_v5.csv", "RSS_v9.csv", "RSS_v11.csv")


df1: (16099, 16)
df2: (16061, 11)
Merged shape: (7038, 26)
Saved as RSS_v11.csv
